# Silver Layer Monitor
Inventory, incoming data pipeline health, and outgoing data status for the LEAGUE_RECORDS.SILVER schema.

In [ ]:
%%sql
USE DATABASE LEAGUE_RECORDS;
USE SCHEMA SILVER;

---
## 1. Object Inventory
Take stock of all objects in the Silver schema: tables, views, streams, tasks, and dynamic tables.

In [ ]:
SHOW TABLES IN SCHEMA LEAGUE_RECORDS.SILVER;

In [ ]:
SHOW VIEWS IN SCHEMA LEAGUE_RECORDS.SILVER;

In [ ]:
SHOW TASKS IN SCHEMA LEAGUE_RECORDS.SILVER;

---
## 2. Incoming Data Monitoring
Monitor the pipeline feeding data into silver: task execution history, stream lag, and row counts.

In [ ]:
SELECT
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME) AS duration_sec,
    ERROR_CODE,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD(
        'hour', -24, CURRENT_TIMESTAMP()
    ),
    RESULT_LIMIT => 100
))
WHERE SCHEMA_NAME = 'SILVER'
ORDER BY NAME, SCHEDULED_TIME DESC;

In [ ]:
SELECT
    NAME AS task_name,
    STATE,
    COUNT(*) AS run_count,
    ROUND(AVG(DATEDIFF('second', SCHEDULED_TIME, COMPLETED_TIME)), 2) AS avg_duration_sec
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 500
))
WHERE SCHEMA_NAME = 'SILVER'
GROUP BY NAME, STATE
ORDER BY NAME, STATE;

In [ ]:
SELECT
    'MATCHES_STM' AS stream_name,
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_STM') AS has_data
UNION ALL
SELECT 'PLAYERS_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.PLAYERS_STM')
UNION ALL
SELECT 'ITEMS_REF_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.ITEMS_REF_STM')
UNION ALL
SELECT 'CHAMPIONS_REF_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.CHAMPIONS_REF_STM')
UNION ALL
SELECT 'INTERVALS_STM',
    SYSTEM$STREAM_HAS_DATA('BRONZE.INTERVALS_STM')
;

In [ ]:
SELECT
    TABLE_NAME,
    ROW_COUNT,
    BYTES,
    ROUND(BYTES / 1024 / 1024, 2) AS size_mb,
    LAST_ALTERED
FROM LEAGUE_RECORDS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'SILVER'
    AND TABLE_TYPE = 'BASE TABLE'
ORDER BY ROW_COUNT DESC;

---
## 3. Outgoing Data Monitoring
Monitor data flowing out of silver into the gold layer: row counts.

In [ ]:
%%sql
-- Compare silver vs gold row counts for key relationships
SELECT
    'MATCHES: Silver vs Gold' AS comparison,
    (SELECT COUNT(*) FROM SILVER.MATCHES) AS silver_rows,
    (SELECT COUNT(*) FROM GOLD.MATCHEND_PIVOT_TEAMSTATS) AS gold_rows,
    (SELECT COUNT(*) FROM SILVER.MATCHES) -
    (SELECT COUNT(*) FROM GOLD.MATCHEND_PIVOT_TEAMSTATS) AS diff
UNION ALL
SELECT
    'PLAYERS: Silver vs Gold',
    (SELECT COUNT(*) FROM SILVER.PLAYERS),
    (SELECT COUNT(*) FROM GOLD.MATCHEND_PLAYER_STATS),
    (SELECT COUNT(*) FROM SILVER.PLAYERS) -
    (SELECT COUNT(*) FROM GOLD.MATCHEND_PLAYER_STATS)
UNION ALL
SELECT
    'CHAMPIONS: Ref vs Overview',
    (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF WHERE CHAMPION_ID != 0),
    (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEWS),
    (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF WHERE CHAMPION_ID != 0) -
    (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEWS)
;